<a href="https://colab.research.google.com/github/niranjanappaji/LLM_Engineering/blob/main/2_LLM_Web_Summarizer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

###
==============================================================================

### End-to-End LLM Web Summarizer & Translator

#### Welcome to this hands-on engineering notebook! This project demonstrates how to:
##### 1. **Ingest & Sanitize Web Data:** Build a hybrid BS4 + Headless Selenium scraper.
##### 2. **Frontier Cloud APIs:** Call OpenAI models using structured system prompts.
##### 3. **Local Transformer Debugging:** Troubleshoot sequence limits in `BART-Large-CNN`.
##### 4. **Modern Local Open Models:** Deploy `Llama 3.1 8B Instruct` using Hugging Face.
==============================================================================

In [ ]:
# ==============================================================================
# [Code] Setup Environment & Dependencies
# ==============================================================================
# Install required libraries in Google Colab / Jupyter environment

!pip install -q requests bs4 selenium openai transformers torch accelerate bitsandbytes huggingface_hub

import os
import re
import torch
import requests
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from IPython.display import Markdown, display

print(f"✅ Setup complete. PyTorch Version: {torch.__version__} | CUDA Available: {torch.cuda.is_available()}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.6/9.6 MB 20.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 511.8/511.8 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.1/131.1 kB 2.8 MB/s eta 0:00:00
✅ Setup complete. PyTorch Version: 2.11.0+cpu | CUDA Available: False


==============================================================================
###### **Step 1: Hybrid Web Ingestion Engine**
###### Web pages consist of static HTML and dynamic JavaScript content.
###### - **Tier 1 (BS4):** Fast HTTP fetch for standard pages.
###### - **Tier 2 (Selenium):** Headless browser fallback for dynamic JS rendering.
==============================================================================

In [ ]:
# ==============================================================================
# [Code] Web Scraper Implementation
# ==============================================================================

def clean_scraped_text(raw_text):
  """Strips excessive whitespace, repetitive menu links, and URL strings."""
  cleaned = re.sub(r'\s+', ' ',raw_text)
  cleaned = re.sub(r'http\S+', ' ', cleaned)
  return cleaned.strip()

def fetch_website_contents_bs4(url):
  """Tier 1: Fast static HTML extraction using requests and BeautifulSoup."""
  try:
    response = requests.get(
        url,
        headers={"User-Agent":"Mozilla/5.0"},
        timeout=5
    )

    soup = BeautifulSoup(response.text, "html.parser")

    # Remove non-content DOM nodes
    for element in soup(["script", "style", "nav", "footer", "header"]):
      element.decompose()

    text = soup.get_text(" ", strip=True)
    cleaned = clean_scraped_text(text)
    return cleaned if len(cleaned) >= 100 else None

  except Exception as e:
    print(f" [BS4 Error]: {e}")
    return None

def fetch_website_contents_selenium(url):
  """Tier 2: Fallback headless browser scraper for dynamic JS pages."""

  options = Options()
  options.add_argument("--headless")
  options.add_argument("--no-sandbox")
  options.add_argument("--disable-dev-shm-usage")

  driver = webdriver.Chrome(options=options)

  try:
    driver.get(url)
    soup = BeautifulSoup(driver.page_source,"html.parser")

    for element in soup(["script", "style", "nav", "footer", "header"]):
      element.decompose()

    text = soup.get_text(" ", strip=True)
    cleaned = clean_scraped_text(text)
    return cleaned if len(cleaned) >= 100 else None

  except Exception as e:
    print(f" [Selenium Error]: {e}")
    return None

  finally:
    driver.quit()

def webscrape(url):
  """Master Ingestion Pipeline."""
  print(f"Ingesting target URL: {url}")
  result = fetch_website_contents_bs4(url)
  if result:
    print("Tier 1 (BS4) successfully retrieved content..")
    return result

  print("Tier 1 Returned thin/empty text. Trying Tier 2 Selenium..")
  return fetch_website_contents_selenium(url)


test_url = "https://en.wikipedia.org/wiki/Singapore"
raw_text = webscrape(test_url)

if raw_text:
    print(f"\n📄 Successfully scraped {len(raw_text)} characters.")
    print(f"Sample: {raw_text[:200]}...")


Ingesting target URL: https://en.wikipedia.org/wiki/Singapore
Tier 1 (BS4) successfully retrieved content..

📄 Successfully scraped 210338 characters.
Sample: Singapore - Wikipedia Jump to content Coordinates : 1°17′N 103°50′E ﻿ / ﻿ 1.283°N 103.833°E ﻿ / 1.283; 103.833 From Wikipedia, the free encyclopedia Island country in Southeast Asia This article is ab...


==============================================================================
##### **Step 2: OpenAI Frontier API Integration**

###### We configure our client dynamically and decouple the system prompt logic
###### to support multiple task execution modes (Summarization vs Translation).
==============================================================================

In [ ]:
# ==============================================================================
# [Code] OpenAI Pipeline
# ==============================================================================

from openai import OpenAI
from google.colab import userdata

def initialize_openai_api_client():

  """Extracts secret API key and initializes OpenAI client."""
  try:
    api_key = userdata.get('OPENAI_API_KEY')
  except Exception:
    api_key = os.environ.get('OPENAI_API_KEY')

  if not api_key:
    print("Please set api_key either in os environment or colab secrets")

  os.environ['OPENAI_API_KEY'] = api_key.strip()
  return OpenAI()

openai = initialize_openai_api_client()


def build_system_prompt(task_mode = "summarize"):

  if task_mode == "translator":
    return (
        """
        You are an expert technical translator. Translate the provided web page text accurately
        into the specified target language while preserving the technical structure and markdown formatting
        """
    )
  return (
      """
      You are an analytical assistant. Summarize the provided webpage content into clear, bulleted markdown key takeaways,
      highlighting the core data points while filtering out navigational fluff.
      """
  )

def process_with_openai(text, task_mode = "summarize", target_language="English", model = "gpt-4o-mini"):

  if not openai:
    return print("Error: OpenAI client not initialized")

  system_prompt = build_system_prompt(task_mode)
  user_prompt = f"Target Language: {target_language}\n\n Content:\n{text[:10000]}"

  response = openai.chat.completions.create(
      model=model,
      messages=[
          {"role":"system", "content":system_prompt},
          {"role":"user", "content":user_prompt}
      ]
                                            )

  return response.choices[0].message.content

if raw_text and openai:

  print("Summarize via OpenAI API...")
  openai_summary = process_with_openai(text = raw_text, task_mode="summarize", target_language="English")
  display(Markdown(openai_summary))

  # print("Translate via OpenAI API...")
  # openai_translate = process_with_openai(text = raw_text, task_mode="translator", target_language="Kannada")
  # display(Markdown(openai_translate))

Summarize via OpenAI API...


# Key Takeaways on Singapore

- **Official Name and Location**
  - Officially named the Republic of Singapore.
  - An island country located in Southeast Asia, approximately 1° north of the equator.

- **Capital and Government**
  - Capital: Singapore (city-state).
  - Government Type: Unitary parliamentary republic.
  - Current President: Tharman Shanmugaratnam.
  - Current Prime Minister: Lawrence Wong.

- **Demographics (2023 Estimates)**
  - Population: Approximately 6.11 million.
  - Ethnic Composition: 
    - 74.3% Chinese
    - 13.5% Malay
    - 9.0% Indian
    - 3.2% Other

- **Official Languages**
  - English, Malay, Mandarin, and Tamil.
  - Malay is recognized as the national language.

- **Religion (Projected 2025)**
  - 30.9% Buddhists
  - 23.9% No religion
  - 17.1% Christians
  - 15.0% Muslims
  - Additional faiths include Taoism and Hinduism.

- **Geography and Area**
  - Total Area: 744.3 km² (287.4 sq mi).
  - Water Coverage: 1.43%.

- **Economics**
  - GDP (PPP) 2026 estimate: $1.063 trillion (34th globally).
  - GDP per capita (PPP) 2026 estimate: $173,708 (highest globally).
  - GDP (nominal) 2026 estimate: $659.572 billion (27th globally).
  - GDP per capita (nominal): $107,758 (4th globally).
  - Gini Index (2023): 43.3, indicating medium inequality.

- **Human Development Index**
  - HDI (2023): 0.946, categorized as very high (13th globally).

- **Cultural Aspects**
  - Recognized as a multicultural society with a strong emphasis on multi-racialism.
  - Known for high standards in education, healthcare, and infrastructure.

- **Historical Background**
  - Established as a trading post by Stamford Raffles in 1819.
  - Became self-governing on June 3, 1959, and independent on August 9, 1965, after separation from Malaysia.
  - Has a complex history involving various ruling empires and colonial influence.

- **Urban Development**
  - High population density, with extensive urban planning that includes green spaces.
  - Often referred to as the "Garden City" due to its greenery and parks.

- **International Relations**
  - Member of ASEAN and various international organizations such as the United Nations and the World Trade Organization.

- **Additional Notes**
  - Singapore is noted for being a tax haven and a major hub for finance and maritime shipping.
  - It enjoys one of the highest life expectancies and lowest levels of corruption in the world.

========================================================================
###### **Step 3: Local Transformer Pipeline (BART-Large-CNN)**
###### When using local task models like `facebook/bart-large-cnn`, passing raw strings
###### longer than 1,024 tokens causes PyTorch embedding out-of-bounds errors.
###### We fix this by explicitly tokenizing and truncating before model generation.
=======================================================================

In [ ]:
# ==============================================================================
# [Code] Local BART Implementation
# ==============================================================================

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

BART_MODEL_NAME = "facebook/bart-large-cnn"

print("Loading BART-large-CNN model into memory..")

bart_tokenizer = AutoTokenizer.from_pretrained(BART_MODEL_NAME)
bart_model = AutoModelForSeq2SeqLM.from_pretrained(BART_MODEL_NAME,
                                                   device_map = "auto")


def summarize_with_bart(text):

  # Step A: Pre-tokenize and cut sequence at exactly 1024 tokens
  inputs = bart_tokenizer(
      text,
      return_tensors="pt",
      Truncation = True,
      max_length = 1024
  ).to(bart_model.device)

  # Step B: Generate output IDs without gradient tracking
  with torch.no_grad():
    summary_ids = bart_model.generate(
        **inputs,
        max_new_tokens=150,
        min_new_tokens=30,
        num_beams=4,
        length_penalty=2.0
    )

  # Step C: Return bart_tokenizer.decode(summary_ids[0], skip_special_tokens=True)
if raw_text:
  print(" Generating Summary using local BART summarizer..")
  bart_summary = summarize_with_bart(raw_text)
  display(Markdown(f"BART Sumamrizer: \n\n{bart_summary}"))


Loading BART-large-CNN model into memory..


config.json:   0%|          | 0.00/1.58k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/511 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

 Generating Summary using local BART summarizer..


[transformers] Both `max_new_tokens` (=150) and `max_length`(=142) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `min_new_tokens` (=30) and `min_length`(=56) seem to have been set. `min_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


BART Sumamrizer: 

None

=========================================================================
###### **Step 4: Compact Local Causal LLM (TinyLlama 1.1B Chat)**
###### `TinyLlama/TinyLlama-1.1B-Chat-v1.0` is an extremely lightweight open-weight
###### model built on Llama 2 architecture. It runs quickly on CPU or low-VRAM
###### GPUs (requiring ~2.2 GB VRAM in FP16/bfloat16) while supporting chat templates.
=========================================================================

In [ ]:
# ==============================================================================
# [Code] Local TinyLlama 1.1B Implementation
# ==============================================================================

from transformers import AutoTokenizer, AutoModelForCausalLM

TINYLLAMA_MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

try:
  print("Loading TinyLlama 1.1B Chat..")
  tinyllama_tokenizer = AutoTokenizer.from_pretrained(TINYLLAMA_MODEL_NAME)
  tinyllama_model = AutoModelForCausalLM.from_pretrained(
      TINYLLAMA_MODEL_NAME,
      torch_dtype=torch.float16,
      device_map="auto")

  def summarizer_with_tinyllama(text):
    """Executes local inference using TinyLlama chat templates."""
    messages = [
        {
            "role":"system",
            "content":"You are an expert research analyst. Summarize the provided webpage content into clear, bulleted markdown key takeaways"
         },
        {
            "role":"user",
            "content":f"Web Content:\n\n{text[:4000]}" # Truncated to fit TinyLlama's 2048 context window
        }
    ]

    prompt = tinyllama_tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tinyllama_tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=2048
    ).to(tinyllama_model.device)

    with torch.no_grad():
      outputs = tinyllama_model.generate(
          **inputs,
          max_new_tokens = 250,
          pad_token_id = tinyllama_tokenizer.eos_token_id
      )

    # Slice tensor to return ONLY generated tokens
    input_length = inputs["input_ids"].shape[1]
    generated_tokens = outputs[0][input_length:]

    return tinyllama_tokenizer.decode(generated_tokens, skip_special_tokens=True)

  if raw_text:
    print("Running local TinyLlama 1.1B Summarizer..")
    tinyllama_summary = summarizer_with_tinyllama(raw_text)
    display(Markdown(tinyllama_summary))

except Exception as e:
  print(f" Error executing TinyLlama : {e}")



Loading TinyLlama 1.1B Chat..


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.29k [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B /  500kB            

tokenizer.model: downloading bytes:           |  0.00B            

...

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

[transformers] Both `max_new_tokens` (=250) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Running local Llama 3.1 8B Instruct..


Singapore is an island country in Southeast Asia that is officially known as the Republic of Singapore. It is located on the southern tip of the Malay Peninsula, bordering the Strait of Malacca to the west, the Singapore Strait to the south, the Riau Islands in Indonesia to the east, and the Straits of Johor along with the State of Johor in Malaysia to the north. Singapore's territory comprises a main island, over 60 satellite islands, and one outlying islet. The country is about one degree of latitude (137 kilometers or 85 miles) north of the equator, and it is bordered by the Strait of Malacca to the west, the Singapore Strait to the south, the Riau Islands in Indonesia to the east, and the Straits of Johor along with the State of Johor in Malaysia to the north. Singapore's contemporary era began in 1819, when Stamford Raffles established Singapore as an entrepôt trading post of the British Empire. In 1867, Singapore came under direct British control as part of